## Structured Streaming Concepts

### In this lesson you:
* Stream data from a file and write it out
* List active streams
* Stop active streams

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Music Log Streaming2") \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
    .getOrCreate()

26/05/18 18:31:17 WARN Utils: Your hostname, codespaces-711bff resolves to a loopback address: 127.0.0.1; using 10.0.13.154 instead (on interface eth0)
26/05/18 18:31:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 18:31:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


`spark.sql.streaming.forceDeleteTempCheckpointLocation` forces the deletion of temporary checkpoint locations when stopping a streaming query.


### The Schema

Every streaming DataFrame must have a schema - the definition of column names and data types. Some sources such as Kafka define the schema for you. In file-based streaming sources, for example, the schema is user-defined.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

music_log_schema = StructType([
    StructField("userId", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("song", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("level", StringType(), True),
    StructField("auth", StringType(), True)
])

### Why are streaming DataFrames unable to infer/read a schema?

The answer:
If you have enough data, you can infer the schema.

If you don't have enough data you run the risk of miss-inferring the schema.
For example, you think you have all integers but the last value contains "1.123" (a float) or "snoopy" (a string).
With a stream, we have to assume we don't have enough data because we are starting with zero records.
And unlike reading from a table or parquet file, there is nowhere from which to "read" the stream's schema.
For this reason, we must specify the schema manually.


### Reading a Stream

The method `SparkSession.readStream` returns a `DataStreamReader` used to configure the stream.

There are a number of key points to the configuration of a `DataStreamReader`:
* The schema
* The type of stream: Files, Kafka, TCP/IP, etc
* Configuration specific to the type of stream
  * For files, the file type, the path to the files, max files, etc...
  * For TCP/IP the server's address, port number, etc...
  * For Kafka the server's address, port, topics, partitions, etc...

In [ ]:
# streaming from the file space
streaming_df = spark.readStream.json("./data/", schema=music_log_schema)

### Streaming DataFrames

Other than the call to `spark.readStream`, it looks just like any other `DataFrame`.

But is it a "streaming" `DataFrame`?

You can differentiate between a "static" and "streaming" `DataFrame` with the following call:

In [ ]:
# Static vs Streaming?
streaming_df.isStreaming

### !!! Unsupported Operations !!!

Most operations on a "streaming" DataFrame are identical to a "static" DataFrame.

**BUT** There are some exceptions to this.

One such example would be to sort our never-ending stream.

In [ ]:
from pyspark.sql.functions import col

try:
  sortedDF = streaming_df.orderBy(col("Recorded_At").desc())
  display(sortedDF)
except:
  print("Sorting is not supported on an unaggregated stream")

Sorting is one of a handful of operations that is either too complex or logically not possible to do with a stream.

For more information on this topic, see the <a href="https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#unsupported-operations" target="_blank">Structured Streaming Programming Guide / Unsupported Operations</a>.

# Writing a Stream

The method `DataFrame.writeStream` returns a `DataStreamWriter` used to configure the output of the stream.

There are a number of parameters to the `DataStreamWriter` configuration:
* Query's name (optional) - This name must be unique among all the currently active queries in the associated SQLContext.
* Trigger (optional) - Default value is `ProcessingTime(0`) and it will run the query as fast as possible.
* Checkpointing directory (optional)
* Output mode
* Output sink
* Configuration specific to the output sink, such as:
  * The host, port and topic of the receiving Kafka server
  * The file format and final destination of files
  * A custom sink via `dsw.foreach(...)`

Once the configuration is completed, we can trigger the job with a call to `dsw.start()`

In [ ]:
import time
from IPython.display import clear_output

# Write to memory sink for debugging within Jupyter
query = streaming_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("streaming_data_view") \
    .start()

try:
    for i in range(10):  # Run for 10 intervals
        # Clear output (to avoid flooding the notebook)

        clear_output(wait=True)
        
        # Show the data in the in-memory table
        display(spark.sql("SELECT * FROM streaming_data_view").show())

        time.sleep(3)  # Pause for a few seconds
finally:
    query.stop()


## Aggregation of the data

### Processing the Stream
You can perform transformations or aggregations on streaming_df just like you would on a static DataFrame. 

For example, if you wanted to count the number of times each song is played:

In [ ]:
from pyspark.sql.functions import window

# Assuming 'timestamp' is the column based on which windowing is required.
# The watermark delays handling any events older than 1 hour past the latest received event.
streaming_data = streaming_df \
    .groupBy("song") \
    .count()

In [ ]:
# Write to memory sink
query = streaming_data.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("song_plays_table")  \
    .start()

try:
    for x in range(10):  # Loop for a number of times to show updates
        spark.sql("SELECT * FROM song_plays_table").show()  # Query the in-memory table
        time.sleep(3)  # Sleep for a few seconds before the next update
finally:
    query.stop()  # Make sure to stop the streaming query to free resources


### Configuring a File Stream

To control how much data is pulled into Spark at once, we can specify the option `maxFilesPerTrigger`.

In our example below, we will be reading in only one file for every trigger interval:

`dsr.option("maxFilesPerTrigger", 1)`

Both the location and file type are specified with the following call, which itself returns a `DataFrame`:

`df = dsr.json(dataPath)`

In [ ]:
dataPath = "./data/"
initial_df = (spark
  .readStream                            # Returns DataStreamReader
  .option("maxFilesPerTrigger", 1)       # Force processing of only 1 file per trigger 
  .schema(music_log_schema)                    # Required for all streaming DataFrames
  .json(dataPath)                        # The stream's source directory and file type
)

And with the initial `DataFrame`, we can apply some transformations:

In [ ]:
streamingDF = (initial_df
  .withColumnRenamed("Index", "User_ID")  # Pick a "better" column name
  .drop("userAgent")                # Remove an unnecessary column
)


### Triggers

The trigger specifies when the system should process the next set of data.

| Trigger Type                           | Example | Notes |
|----------------------------------------|-----------|-------------|
| Unspecified                            |  | _DEFAULT_- The query will be executed as soon as the system has completed processing the previous query |
| Fixed interval micro-batches           | `dsw.trigger(Trigger.ProcessingTime("6 hours"))` | The query will be executed in micro-batches and kicked off at the user-specified intervals |
| One-time micro-batch                   | `dsw.trigger(Trigger.Once())` | The query will execute _only one_ micro-batch to process all the available data and then stop on its own |
| Continuous w/fixed checkpoint interval | `dsw.trigger(Trigger.Continuous("1 second"))` | The query will be executed in a low-latency, continuous processing mode. _EXPERIMENTAL_ in 2.3.2 |

In the example below, you will be using a fixed interval of 3 seconds:

`dsw.trigger(Trigger.ProcessingTime("3 seconds"))`

### Checkpointing

A <b>checkpoint</b> stores the current state of your streaming job to a reliable storage system such as Blob Storage, File System or HDFS. It does not store the state of your streaming job to the local file system of any node in your cluster. 

Together with write ahead logs, a terminated stream can be restarted and it will continue from where it left off.

To enable this feature, you only need to specify the location of a checkpoint directory:

`dsw.option("checkpointLocation", checkpointPath)`


Points to consider:
* If you do not have a checkpoint directory, when the streaming job stops, you lose all state around your streaming job and upon restart, you start from scratch.
* For some sinks, you will get an error if you do not specify a checkpoint directory:<br/>
`analysisException: 'checkpointLocation must be specified either through option("checkpointLocation", ...)..`
* Also note that every streaming job should have its own checkpoint directory: no sharing.

### Output Modes

| Mode   | Example | Notes |
| ------------- | ----------- | --------- |
| **Complete** | `dsw.outputMode("complete")` | The entire updated Result Table is written to the sink. The individual sink implementation decides how to handle writing the entire table. |
| **Append** | `dsw.outputMode("append")`     | Only the new rows appended to the Result Table since the last trigger are written to the sink. |
| **Update** | `dsw.outputMode("update")`     | Only the rows in the Result Table that were updated since the last trigger will be outputted to the sink. Since Spark 2.1.1 |

In the example below, we are writing to a Parquet directory which only supports the `append` mode: 

```
dsw.outputMode("append")
```



### Output Sinks

`DataStreamWriter.format` accepts the following values, among others:

| Output Sink | Example                                          | Notes |
| ----------- | ------------------------------------------------ | ----- |
| **File**    | `dsw.format("parquet")`, `dsw.format("csv")`...  | Dumps the Result Table to a file. Supports Parquet, json, csv, etc.|
| **Kafka**   | `dsw.format("kafka")`      | Writes the output to one or more topics in Kafka |
| **Console** | `dsw.format("console")`    | Prints data to the console (useful for debugging) |
| **Memory**  | `dsw.format("memory")`     | Updates an in-memory table, which can be queried through Spark SQL or the DataFrame API |
| **foreach** | `dsw.foreach(writer: ForeachWriter)` | This is your "escape hatch", allowing you to write your own type of sink. |
| **Delta**    | `dsw.format("delta")`     | A proprietary sink |

In the example below, we will be appending files to a Parquet file and specifying its location with this call:

`dsw.format("parquet").start(outputPathDir)`

<h2> Let's Do Some Streaming</h2>

In the cell below, we write data from a streaming query to `outputPathDir`. 

There are a couple of things to note below:
1. We are giving the query a name via the call to `queryName` . We can use this later to reference the query by name.
1. Spark begins running jobs once we call `start`. This is equivilent to calling an action on a "static" DataFrame.
1. The call to `start` returns a `StreamingQuery` object. We can use this to interact with the running query.

In [ ]:
basePath = "./streaming-app" # A working directory for our streaming app
outputPathDir = basePath + "/output.parquet"                  # A subdirectory for our output
checkpointPath = basePath + "/checkpoint"                     # A subdirectory for our checkpoint & W-A logs

streamingQuery = (streaming_df                                 # Start with our "streaming" DataFrame
  .writeStream                                                # Get the DataStreamWriter
  .queryName("stream_1p")                                     # Name the query
  .trigger(processingTime="3 seconds")                        # Configure for a 3-second micro-batch
  .format("parquet")                                          # Specify the sink type, a Parquet file
  .option("checkpointLocation", checkpointPath)               # Specify the location of checkpoint files & W-A logs
  .outputMode("append")                                       # Write only new data to the "file"
  .start(outputPathDir)                                       # Start the job, writing to the specified directory
)


<h2> Managing Streaming Queries</h2>

When a query is started, the `StreamingQuery` object can be used to monitor and manage the query.

| Method    |  Description |
| ----------- | ------------------------------- |
|`id`| get unique identifier of the running query that persists across restarts from checkpoint data |
|`runId`| get unique id of this run of the query, which will be generated at every start/restart |
|`name`| get name of the auto-generated or user-specified name |
|`explain()`| print detailed explanations of the query |
|`stop()`| stop query |
|`awaitTermination()`| block until query is terminated, with stop() or with error |
|`exception`| exception if query terminated with error |
|`recentProgress`| array of most recent progress updates for this query |
|`lastProgress`| most recent progress update of this streaming query |

In [ ]:
streamingQuery.recentProgress

Additionally, we can iterate over a list of active streams:

In [ ]:
for s in spark.streams.active:         # Iterate over all streams
  print("{}: {}".format(s.id, s.name)) # Print the stream's id and name

The code below stops the `streamingQuery` defined above and introduces `awaitTermination()`

`awaitTermination()` will block the current thread
* Until the stream stops or 
* Until the specified timeout elapses

In [ ]:
streamingQuery.awaitTermination(5)      # Stream for another 5 seconds while the current thread blocks
streamingQuery.stop()                   # Stop the stream

#### One Gotcha!
When you pass a "streaming" `DataFrame` to `display`:
* A "memory" sink is being used
* The output mode is complete
* The query name is specified with the `streamName` parameter
* The trigger is specified with the `trigger` parameter
* The checkpointing location is specified with the `checkpointLocation`

Using the value passed to `streamName` in the call to `display`, we can programatically access this specific stream:

In [ ]:
print("Looking for {}".format("stream_1p"))
myStream = "stream_1p"
for stream in spark.streams.active:      # Loop over all active streams
  if stream.name == myStream:            # Single out "streamWithTimestamp"
    print("Found {} ({})".format(stream.name, stream.id)) 

Stop all remaining streams.

In [ ]:
for s in spark.streams.active:
  s.stop()

In [ ]:
spark.stop()

## End-to-end Fault Tolerance

Structured Streaming ensures end-to-end exactly-once fault-tolerance guarantees through _checkpointing_ and <a href="https://en.wikipedia.org/wiki/Write-ahead_logging" target="_blank">Write Ahead Logs</a>.

Structured Streaming sources, sinks, and the underlying execution engine work together to track the progress of stream processing. If a failure occurs, the streaming engine attempts to restart and/or reprocess the data.

This approach _only_ works if the streaming source is replayable. To ensure fault-tolerance, Structured Streaming assumes that every streaming source has offsets, akin to:

* <a target="_blank" href="https://kafka.apache.org/documentation/#intro_topics">Kafka message offsets</a>

At a high level, the underlying streaming mechanism relies on a couple approaches:

* First, Structured Streaming uses checkpointing and write-ahead logs to record the offset range of data being processed during each trigger interval.
* Next, the streaming sinks are designed to be _idempotent_—that is, multiple writes of the same data (as identified by the offset) do _not_ result in duplicates being written to the sink.

Taken together, replayable data sources and idempotent sinks allow Structured Streaming to ensure **end-to-end, exactly-once semantics** under any failure condition.


<h2> Summary</h2>

We use `readStream` to read streaming input from a variety of input sources and create a DataFrame.

Nothing happens until we invoke `writeStream` or `display`. 

Using `writeStream` we can write to a variety of output sinks. Using `display` we draw LIVE bar graphs, charts and other plot types in the notebook.


<h2> Review Questions</h2>

**Q:** What do `readStream` and `writeStream` do?<br>
**A:** `readStream` creates a streaming DataFrame.<br>`writeStream` sends streaming data to a directory or other type of output sink.



**Q:** When you do a write stream command, what does this option do `outputMode("append")` ?<br>
**A:** This option takes on the following values and their respective meanings:
* <b>append</b>: add only new records to output sink
* <b>complete</b>: rewrite full output - applicable to aggregations operations
* <b>update</b>: update changed records in place

**Q:** What happens if you do not specify `option("checkpointLocation", pointer-to-checkpoint directory)`?<br>
**A:** When the streaming job stops, you lose all state around your streaming job and upon restart, you start from scratch.

**Q:** How do you view the list of active streams?<br>
**A:** Invoke `spark.streams.active`.

**Q:** How do you verify whether `streamingQuery` is running (boolean output)?<br>
**A:** Invoke `spark.streams.get(streamingQuery.id).isActive`.